<a href="https://colab.research.google.com/github/sadwi18/24P31A4236_DriveReady_OPC_2026/blob/main/task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# SetUp
!pip install -q langchain langchain-groq gspread pydantic

import os
import gspread
from google.auth import default
from google.colab import auth, userdata
from langchain_groq import ChatGroq

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_retries=3,
)

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

SHEET_NAME = "ShopSphere Orders"
try:
    sh = gc.open(SHEET_NAME)
    ws = sh.sheet1
except gspread.SpreadsheetNotFound:
    sh = gc.create(SHEET_NAME)
    ws = sh.sheet1
    ws.append_row(["customer_name", "product", "qty", "city", "cod", "total"])

print("Sheet ready:", sh.url)

Sheet ready: https://docs.google.com/spreadsheets/d/1oT4A2ovMw_ks3ufndRYEKEv_Zd8HeL4u85GmpZrTu-4


In [3]:
# Data, the Order model and three tools
from typing import Literal
from langchain.tools import tool
from pydantic import BaseModel, Field

CATALOG = {
    "wireless earbuds": 1499,
    "smartwatch": 2499,
    "laptop backpack": 999,
}

STOCK = {
    "wireless earbuds": 18,
    "smartwatch": 7,
    "laptop backpack": 25,
}


class ShopSphereOrder(BaseModel):
    """A confirmed order taken from a customer's chat message."""
    customer_name: str = Field(description="Customer's first name")
    product: Literal["wireless earbuds", "smartwatch", "laptop backpack"] = Field(
        description="ShopSphere catalog item"
    )
    qty: int = Field(description="How many units ordered", ge=1)
    city: str = Field(description="Delivery city as the customer gave it")
    cod: bool = Field(description="True if Cash on Delivery is requested, False otherwise")


@tool
def get_product_price(product_name: str, quantity: int = 1) -> str:
    """Calculate the total price in rupees (INR) for a quantity of one ShopSphere product.

    Args:
        product_name: Catalog item, e.g. 'wireless earbuds', 'smartwatch', or 'laptop backpack'
        quantity: How many units the customer wants (must be >= 1)
    """
    key = product_name.lower().strip()
    if key not in CATALOG:
        return f"'{product_name}' is not in our catalog. Available items: {', '.join(CATALOG)}."
    if quantity < 1:
        return "Quantity must be at least 1."
    total = CATALOG[key] * quantity
    shipping_note = "free shipping" if total >= 1500 else "3-5 business days standard shipping"
    return f"{quantity} x {key} = Rs {total} ({shipping_note})."


@tool
def check_stock(product_name: str) -> str:
    """Check current stock availability for a ShopSphere catalog item.

    Args:
        product_name: Catalog item, e.g. 'wireless earbuds', 'smartwatch', or 'laptop backpack'
    """
    key = product_name.lower().strip()
    if key not in STOCK:
        return f"'{product_name}' is not in our catalog. Available items: {', '.join(CATALOG)}."
    units = STOCK[key]
    if units <= 0:
        return f"{key.title()} is currently out of stock."
    return f"{key.title()} is in stock ({units} units available)."


@tool
def log_order(customer_name: str, product: str, qty: int, city: str, cod: bool) -> str:
    """Save a confirmed ShopSphere order to the Google Sheet.
    Use only when the customer has given customer_name, product, qty, city, and payment mode (COD/online)."""
    try:
        order = ShopSphereOrder(
            customer_name=customer_name.strip(),
            product=product.lower().strip(),
            qty=qty,
            city=city.strip(),
            cod=cod,
        )
    except Exception as e:
        return f"Could not save the order: {e}. Ask the customer to correct it."

    if STOCK.get(order.product, 0) < order.qty:
        return f"Only {STOCK.get(order.product, 0)} units of {order.product} available right now."

    total = CATALOG[order.product] * order.qty
    ws.append_row([
        order.customer_name,
        order.product,
        order.qty,
        order.city,
        "yes" if order.cod else "no",
        total,
    ])
    return (
        f"Saved: {order.qty} x {order.product} for {order.customer_name} "
        f"to {order.city} (COD: {'yes' if order.cod else 'no'}), Rs {total}."
    )

In [4]:
# The agent
extractor = llm.with_structured_output(ShopSphereOrder)

messy_orders = [
    "hi im sadwika, need 2 wireless earbuds to kakinada cod pls",
    "hey this is ravi, book 1 smartwatch for hyderabad, online payment",
    "hello ananya here!! 3 laptop backpacks for vizag cash on delivery",
    "plz send 1 wireless earbuds to vijayawada, name kiran, cod",
    "rohit here, order 2 smartwatches for bengaluru, upi payment",
]

for msg in messy_orders:
    parsed = extractor.invoke(msg)
    print("Message:", msg)
    print("Parsed :", parsed.model_dump())
    print("-" * 50)

# Break-test: vague tool name & docstring
@tool
def tool1(x: str) -> str:
    """Does stuff."""
    return check_stock.func(x)


from langchain.agents import create_agent

break_agent = create_agent(
    model=llm,
    tools=[get_product_price, tool1, log_order],
    system_prompt="You are the ShopSphere customer support assistant.",
)
break_res = break_agent.invoke({
    "messages": [{"role": "user", "content": "Are smartwatches in stock right now?"}]
})
print("\nBreak-test final reply:", break_res["messages"][-1].content)

Message: hi im sadwika, need 2 wireless earbuds to kakinada cod pls
Parsed : {'customer_name': 'sadwika', 'product': 'wireless earbuds', 'qty': 2, 'city': 'kakinada', 'cod': True}
--------------------------------------------------
Message: hey this is ravi, book 1 smartwatch for hyderabad, online payment
Parsed : {'customer_name': 'Ravi', 'product': 'smartwatch', 'qty': 1, 'city': 'Hyderabad', 'cod': False}
--------------------------------------------------
Message: hello ananya here!! 3 laptop backpacks for vizag cash on delivery
Parsed : {'customer_name': 'ananya', 'product': 'laptop backpack', 'qty': 3, 'city': 'vizag', 'cod': True}
--------------------------------------------------
Message: plz send 1 wireless earbuds to vijayawada, name kiran, cod
Parsed : {'customer_name': 'kiran', 'product': 'wireless earbuds', 'qty': 1, 'city': 'vijayawada', 'cod': True}
--------------------------------------------------
Message: rohit here, order 2 smartwatches for bengaluru, upi payment
Parse

In [5]:
# Main ShopSphere Agent & Chat Loop
SYSTEM = """ROLE:   You are the helpful customer support assistant for ShopSphere Online Store.
FACTS:  7-day return window for unused items. Shipping is 3-5 business days; free on orders Rs 1500+.
        Payments accepted: UPI, cards, and Cash on Delivery (COD).
RULES:  Always use get_product_price for prices/totals and check_stock for stock; never guess.
        Before log_order, confirm customer_name, product, qty, city, and COD preference; ask if missing.
        After saving, read the order back with the total.
FORMAT: Under 60 words, friendly, matching the customer's language (English or Telugu)."""

agent = create_agent(
    model=llm,
    tools=[get_product_price, check_stock, log_order],
    system_prompt=SYSTEM,
)

history = []


def chat(text: str) -> str:
    global history
    if not text.strip():
        return "Please type a question or order request."
    try:
        result = agent.invoke({"messages": history + [{"role": "user", "content": text}]})
        history = result["messages"]
        return history[-1].content
    except Exception as e:
        print("Agent error:", e)
        return "Sorry, I'm a little busy. Please try again in a minute."


print(chat("Are wireless earbuds in stock? How much for 2?"))
print(chat("I'm Sadwika. Book 2 wireless earbuds for Kakinada with COD."))

Yes, wireless earbuds are in stock (18 units available). Two pairs cost Rs 2998, and shipping is free (orders over Rs 1500). Let me know if you’d like to place the order!
Your order for 2 wireless earbuds to Kakinada (COD) is confirmed. Total Rs 2998. Thank you, Sadwika!


In [6]:
# Trace Inspection & Call Counts
model_calls = 0
tool_calls = 0

for m in history:
    m.pretty_print()
    if getattr(m, "type", "") == "ai":
        model_calls += 1
        tool_calls += len(getattr(m, "tool_calls", []) or [])

print(f"\nSummary -> Model calls: {model_calls} | Tool calls: {tool_calls}")

================================ Human Message =================================

Are wireless earbuds in stock? How much for 2?
================================== Ai Message ==================================
Tool Calls:
  check_stock (fc_8523f5a5-c810-4658-86e6-d693d55c7a52)
 Call ID: fc_8523f5a5-c810-4658-86e6-d693d55c7a52
  Args:
    product_name: wireless earbuds
================================= Tool Message =================================
Name: check_stock

Wireless Earbuds is in stock (18 units available).
================================== Ai Message ==================================
Tool Calls:
  get_product_price (fc_15241d2a-eafc-48c1-9fc7-9a111917963b)
 Call ID: fc_15241d2a-eafc-48c1-9fc7-9a111917963b
  Args:
    product_name: wireless earbuds
    quantity: 2
================================= Tool Message =================================
Name: get_product_price

2 x wireless earbuds = Rs 2998 (free shipping).
================================== Ai Message ==============